# SISTEMA ESPECIALISTA COM APRENDIZADO SUPERVISIONADO
## Conformidade de Nota Fiscal antes da Liquidação

**Aluno:** Pedro Freire de Farias  
**Disciplina:** Inteligência Artificial, Linguagem e Autômatos  
**Professora:** Kadidja Valéria

### Objetivo do trabalho

Desenvolver um sistema didático que utilize **aprendizado de máquina** e **regras de produção** para analisar informações de notas fiscais e sugerir o encaminhamento administrativo adequado antes da liquidação.

O projeto terá quatro partes principais:
1. Criar uma base de dados fictícia e explicar como os registros foram classificados.
2. Treinar uma árvore de decisão para aprender padrões presentes nos dados.
3. Avaliar o modelo usando acurácia e matriz de confusão.
4. Criar regras de produção que expliquem a recomendação final.

> **Aviso:** este é um protótipo acadêmico. Os registros são simulados e as regras representam o fluxo simplificado descrito para este trabalho. O sistema não substitui a conferência humana, as normas vigentes nem os procedimentos oficiais da organização.

## Célula 01 — Preparação do ambiente

Nesta primeira etapa, importamos as bibliotecas que serão utilizadas. Não é necessário instalar nada, pois essas bibliotecas normalmente já estão disponíveis no Google Colab.

In [ ]:
# Célula 01: importação das bibliotecas

import pandas as pd  # Organiza os dados em tabelas.
import matplotlib.pyplot as plt  # Cria gráficos.

from sklearn.model_selection import train_test_split  # Separa os dados em treino e teste.
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text  # Cria e mostra a árvore de decisão.
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report  # Avalia o modelo.

# Mostra uma mensagem para confirmar que as bibliotecas foram carregadas.
print("Bibliotecas carregadas com sucesso!")

## Célula 02 — Definição do problema e das informações de entrada

O sistema recebe informações básicas da conferência de uma nota fiscal.

- `tipo_empenho`: informa se o empenho é ordinário, global ou estimativo.
- `valor_confere`: indica se o valor da nota está correto.
- `empenho_confere`: indica se a nota corresponde ao empenho.
- `saldo_suficiente`: indica se há saldo suficiente.
- `certidoes_validas`: indica se as certidões verificadas estão válidas.
- `ateste_recebimento`: indica se existe ateste do recebimento do material ou serviço.
- `ordenador_assinou`: indica se o Ordenador de Despesas (OD) já assinou.

O resultado esperado será uma destas três opções:
- **Devolver ao gestor**
- **Encaminhar ao ordenador**
- **Encaminhar para liquidação**

A ausência da assinatura do OD, sozinha, não significa que a nota deve ser devolvida ao gestor. Se as conferências estiverem corretas, a nota segue para assinatura.

## Célula 03 — Criação do dataset rotulado

A base contém **36 registros fictícios**, divididos igualmente entre as três classificações (12 registros por classe).

**Origem:** exemplos criados para fins acadêmicos, com base no fluxo simplificado de conferência explicado para o projeto. Não foram utilizados dados reais de processos ou fornecedores.

**Critério de rotulagem:**
- Se houver alguma inconsistência nas conferências, o resultado será `Devolver ao gestor`.
- Se todas as conferências estiverem corretas e o OD ainda não tiver assinado, o resultado será `Encaminhar ao ordenador`.
- Se todas as conferências estiverem corretas e o OD já tiver assinado, o resultado será `Encaminhar para liquidação`.

Os registros foram montados manualmente para que seja possível entender de onde veio cada classificação.

In [ ]:
# Célula 03: criação dos registros fictícios

# Cada lista representa uma nota fiscal simulada.
# A última informação de cada linha é a classificação esperada.

dados = [
    # Registros com alguma inconsistência: devem voltar ao gestor.
    ["Ordinário", True, False, True, True, True, False, "Devolver ao gestor"],
    ["Global", False, True, True, True, True, False, "Devolver ao gestor"],
    ["Estimativo", True, True, False, True, True, False, "Devolver ao gestor"],
    ["Ordinário", True, True, True, False, True, False, "Devolver ao gestor"],
    ["Global", False, True, True, True, True, True, "Devolver ao gestor"],
    ["Estimativo", True, False, True, True, True, True, "Devolver ao gestor"],
    ["Global", True, True, False, True, True, True, "Devolver ao gestor"],
    ["Ordinário", True, True, True, True, False, False, "Devolver ao gestor"],
    ["Estimativo", False, True, True, True, True, False, "Devolver ao gestor"],
    ["Global", True, False, True, True, True, False, "Devolver ao gestor"],
    ["Ordinário", True, True, False, True, True, True, "Devolver ao gestor"],
    ["Estimativo", True, True, True, False, True, False, "Devolver ao gestor"],

    # Registros corretos que ainda aguardam a assinatura do OD.
    ["Global", True, True, True, True, True, False, "Encaminhar ao ordenador"],
    ["Estimativo", True, True, True, True, True, False, "Encaminhar ao ordenador"],
    ["Ordinário", True, True, True, True, True, False, "Encaminhar ao ordenador"],
    ["Global", True, True, True, True, True, False, "Encaminhar ao ordenador"],
    ["Estimativo", True, True, True, True, True, False, "Encaminhar ao ordenador"],
    ["Ordinário", True, True, True, True, True, False, "Encaminhar ao ordenador"],
    ["Global", True, True, True, True, True, False, "Encaminhar ao ordenador"],
    ["Estimativo", True, True, True, True, True, False, "Encaminhar ao ordenador"],
    ["Ordinário", True, True, True, True, True, False, "Encaminhar ao ordenador"],
    ["Global", True, True, True, True, True, False, "Encaminhar ao ordenador"],
    ["Estimativo", True, True, True, True, True, False, "Encaminhar ao ordenador"],
    ["Ordinário", True, True, True, True, True, False, "Encaminhar ao ordenador"],

    # Registros corretos que já possuem a assinatura do OD.
    ["Ordinário", True, True, True, True, True, True, "Encaminhar para liquidação"],
    ["Global", True, True, True, True, True, True, "Encaminhar para liquidação"],
    ["Estimativo", True, True, True, True, True, True, "Encaminhar para liquidação"],
    ["Ordinário", True, True, True, True, True, True, "Encaminhar para liquidação"],
    ["Global", True, True, True, True, True, True, "Encaminhar para liquidação"],
    ["Estimativo", True, True, True, True, True, True, "Encaminhar para liquidação"],
    ["Ordinário", True, True, True, True, True, True, "Encaminhar para liquidação"],
    ["Global", True, True, True, True, True, True, "Encaminhar para liquidação"],
    ["Estimativo", True, True, True, True, True, True, "Encaminhar para liquidação"],
    ["Ordinário", True, True, True, True, True, True, "Encaminhar para liquidação"],
    ["Global", True, True, True, True, True, True, "Encaminhar para liquidação"],
    ["Estimativo", True, True, True, True, True, True, "Encaminhar para liquidação"]
]

# Definimos os nomes das colunas na mesma ordem das informações de cada registro.
colunas = [
    "tipo_empenho",
    "valor_confere",
    "empenho_confere",
    "saldo_suficiente",
    "certidoes_validas",
    "ateste_recebimento",
    "ordenador_assinou",
    "resultado"
]

# Transformamos a lista de registros em uma tabela.
df = pd.DataFrame(dados, columns=colunas)

# Exibimos a tabela completa para conferir os registros.
display(df)

## Célula 04 — Verificação da base de dados

Antes de treinar o modelo, verificamos:
- Quantas linhas e colunas existem.
- Quantos registros pertencem a cada classificação.
- Se há informações vazias.

Essa conferência ajuda a encontrar erros simples na preparação dos dados.

In [ ]:
# Célula 04: conferência do dataset

# Mostra a quantidade de linhas e colunas da tabela.
print("Tamanho do dataset:", df.shape)

# Conta quantos registros existem em cada classificação.
print("\nQuantidade de registros por resultado:")
print(df["resultado"].value_counts())

# Conta quantos valores vazios existem em cada coluna.
print("\nValores vazios por coluna:")
print(df.isnull().sum())

## Célula 05 — Regras do conhecimento e justificativa

As regras abaixo representam a lógica simplificada utilizada neste protótipo. Elas foram definidas a partir do fluxo de trabalho descrito para a atividade e devem ser conferidas com os procedimentos e normas aplicáveis antes de qualquer uso real.

**R1 — Conferência do valor:** se o valor da nota fiscal não conferir, devolver ao gestor para correção.

**R2 — Demais conferências:** se o empenho não corresponder, o saldo for insuficiente, as certidões não estiverem válidas ou faltar o ateste, devolver ao gestor para regularização.

**R3 — Aguardando assinatura:** se todas as conferências estiverem corretas e o OD ainda não tiver assinado, encaminhar ao ordenador para assinatura.

**R4 — Assinatura realizada:** se todas as conferências estiverem corretas e o OD já tiver assinado, encaminhar para liquidação.

**Origem e justificativa:** as regras foram elaboradas para este projeto com base no fluxo administrativo informado pelo aluno. Elas não foram extraídas de uma norma específica e não substituem a validação pelo setor responsável.

## Célula 06 — Preparação das entradas e do resultado esperado

O modelo precisa receber as informações de entrada separadas do resultado que queremos prever.

- `X` terá os dados da nota fiscal.
- `y` terá a classificação correta de cada registro.

Também transformamos o tipo de empenho em colunas numéricas. Isso é necessário porque o classificador trabalha com números, e não diretamente com palavras.

In [ ]:
# Célula 06: preparação dos dados para o modelo

# Separa as colunas de entrada e remove a coluna que contém a resposta.
X = df.drop("resultado", axis=1)

# Guarda as classificações que o modelo deverá aprender a prever.
y = df["resultado"]

# Transforma o tipo de empenho em colunas numéricas.
X = pd.get_dummies(X, columns=["tipo_empenho"], dtype=int)

# Mostra como ficaram as informações de entrada.
display(X.head())

# Mostra os primeiros resultados esperados.
print("Primeiras classificações:")
print(y.head())

## Célula 07 — Separação entre treinamento e teste

Dividimos a base em dois grupos:
- **75% para treinamento:** exemplos usados para ensinar o modelo.
- **25% para teste:** exemplos reservados para avaliar o modelo.

O parâmetro `stratify=y` mantém a proporção das três classificações nos dois grupos. O `random_state=42` faz com que a divisão seja reproduzida da mesma forma quando o notebook for executado novamente.

In [ ]:
# Célula 07: divisão dos dados

# Separa 75% dos registros para treinamento e 25% para teste.
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,                  # Informações de entrada.
    y,                  # Classificações corretas.
    test_size=0.25,     # Reserva 25% dos registros para teste.
    random_state=42,    # Mantém a divisão reproduzível.
    stratify=y          # Mantém a proporção das classes.
)

# Mostra quantos registros ficaram em cada grupo.
print("Registros para treinamento:", len(X_treino))
print("Registros para teste:", len(X_teste))

# Mostra a quantidade de cada classificação no grupo de teste.
print("\nClassificações no grupo de teste:")
print(y_teste.value_counts())

## Célula 08 — Treinamento da árvore de decisão

Agora treinamos um classificador chamado **árvore de decisão**.

De forma simples, ele aprende a fazer perguntas sobre os dados, como:
- O valor da nota confere?
- O empenho confere?
- O OD já assinou?

Com base nas respostas, a árvore aprende caminhos que levam a uma classificação.

In [ ]:
# Célula 08: treinamento do classificador

# Cria a árvore de decisão com uma profundidade limitada para facilitar a leitura.
modelo = DecisionTreeClassifier(
    max_depth=4,       # Limita a quantidade de níveis da árvore.
    random_state=42    # Mantém o resultado reproduzível.
)

# Ensina o modelo usando apenas os dados de treinamento.
modelo.fit(X_treino, y_treino)

# Exibe uma mensagem confirmando o treinamento.
print("Modelo treinado com sucesso!")

## Célula 09 — Avaliação do modelo

Nesta etapa, o modelo faz previsões usando os dados de teste, que ficaram separados do treinamento.

Depois comparamos as previsões com as classificações corretas:
- **Acurácia:** percentual de previsões corretas.
- **Matriz de confusão:** mostra quantos registros de cada classe foram classificados corretamente ou confundidos com outra classe.
- **Relatório de classificação:** apresenta precisão, revocação (recall) e F1-score por classe.

Como os dados são fictícios e pequenos, as métricas servem apenas para demonstrar o funcionamento do método.

In [ ]:
# Célula 09: previsões e métricas

# Faz previsões para os registros que ficaram separados para teste.
previsoes = modelo.predict(X_teste)

# Calcula a proporção de previsões corretas.
acuracia = accuracy_score(y_teste, previsoes)

# Exibe a acurácia em porcentagem.
print(f"Acurácia do modelo: {acuracia * 100:.2f}%")

# Mostra outras métricas para cada classificação.
print("\nRelatório de classificação:")
print(classification_report(y_teste, previsoes, zero_division=0))

# Calcula a matriz de confusão usando a ordem das classes aprendidas pelo modelo.
matriz = confusion_matrix(y_teste, previsoes, labels=modelo.classes_)

# Desenha a matriz de confusão para facilitar a leitura.
visualizacao = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=modelo.classes_
)

# Exibe o gráfico da matriz.
visualizacao.plot(xticks_rotation=25)
plt.title("Matriz de confusão - Notas fiscais")
plt.show()

**Como interpretar o resultado:** na execução de conferência deste notebook, a acurácia foi de aproximadamente **77,78%** (7 acertos em 9 registros de teste). A matriz também mostra que dois registros classificados como "Devolver ao gestor" foram previstos pelo modelo como "Encaminhar ao ordenador".

Isso não significa que as regras administrativas estejam erradas. Significa que, com poucos exemplos e uma árvore limitada a quatro níveis, o modelo ainda comete erros. Por isso o protótipo mostra a previsão do modelo separadamente e mantém a recomendação final sob as regras explícitas do sistema especialista.

## Célula 10 — Visualização da árvore de decisão

O gráfico abaixo mostra as perguntas que a árvore aprendeu durante o treinamento.

Cada divisão representa uma condição dos dados. As folhas finais indicam a classificação prevista pelo modelo.

In [ ]:
# Célula 10: desenho da árvore de decisão

# Define o tamanho da figura para que os textos fiquem mais legíveis.
plt.figure(figsize=(18, 9))

# Desenha a árvore treinada.
plot_tree(
    modelo,
    feature_names=X.columns,  # Mostra o nome das informações de entrada.
    class_names=modelo.classes_,  # Mostra os nomes das classificações.
    filled=True,  # Usa cores para diferenciar as classes.
    rounded=True,  # Deixa os blocos com cantos arredondados.
    fontsize=8  # Ajusta o tamanho do texto.
)

# Adiciona um título ao gráfico.
plt.title("Árvore de decisão - Conformidade de nota fiscal")

# Exibe o gráfico.
plt.show()

## Célula 11 — Regras aprendidas pela árvore

Além do desenho, podemos visualizar em texto as condições que a árvore aprendeu.

Essas condições são resultado do treinamento. Elas não são as mesmas regras administrativas escritas manualmente na seção anterior.

In [ ]:
# Célula 11: exibição das condições aprendidas

# Converte a árvore treinada em uma explicação textual.
regras_aprendidas = export_text(
    modelo,
    feature_names=list(X.columns)
)

# Exibe as condições encontradas pelo modelo.
print(regras_aprendidas)

## Célula 12 — Construção do motor de inferência

Agora criamos o componente de sistema especialista baseado nas quatro regras definidas anteriormente.

O motor recebe os dados de uma nota fiscal, verifica as condições em uma ordem de prioridade e retorna:
- A regra aplicada.
- A recomendação.
- Uma explicação simples do motivo.

A primeira inconsistência encontrada já é suficiente para indicar a devolução ao gestor. Se não houver inconsistências, o sistema verifica a assinatura do OD.

In [ ]:
# Célula 12: função que aplica as regras do sistema especialista

def aplicar_regras(caso):
    # Regra R1: se o valor da nota não conferir, devolver ao gestor.
    if not caso["valor_confere"]:
        return {
            "regra": "R1",
            "recomendacao": "Devolver ao gestor",
            "explicacao": "O valor da nota fiscal precisa ser conferido."
        }

    # Regra R2: verifica as demais condições da conferência.
    elif (
        not caso["empenho_confere"]
        or not caso["saldo_suficiente"]
        or not caso["certidoes_validas"]
        or not caso["ateste_recebimento"]
    ):
        return {
            "regra": "R2",
            "recomendacao": "Devolver ao gestor",
            "explicacao": "Existe uma pendência no empenho, saldo, certidões ou ateste."
        }

    # Regra R3: se tudo estiver correto, mas faltar a assinatura do OD.
    elif not caso["ordenador_assinou"]:
        return {
            "regra": "R3",
            "recomendacao": "Encaminhar ao ordenador",
            "explicacao": "As conferências estão corretas e a nota aguarda a assinatura do OD."
        }

    # Regra R4: se todas as conferências estiverem corretas e o OD já assinou.
    else:
        return {
            "regra": "R4",
            "recomendacao": "Encaminhar para liquidação",
            "explicacao": "As conferências estão corretas e a assinatura do OD foi realizada."
        }

## Célula 13 — Integração do modelo com as regras

Nesta etapa, juntamos os dois componentes:
1. A árvore de decisão apresenta uma previsão baseada nos exemplos de treinamento.
2. O motor de inferência aplica as regras administrativas definidas no projeto e apresenta a recomendação acompanhada de uma justificativa.

A recomendação final exibida pelo protótipo será a produzida pelo motor de regras. A previsão do modelo será mostrada para comparação.

In [ ]:
# Célula 13: função que consulta o modelo e o motor de regras

def consultar_sistema(caso):
    # Organiza os dados recebidos em uma tabela com uma única linha.
    entrada = pd.DataFrame([caso], columns=[
        "tipo_empenho",
        "valor_confere",
        "empenho_confere",
        "saldo_suficiente",
        "certidoes_validas",
        "ateste_recebimento",
        "ordenador_assinou"
    ])

    # Transforma o tipo de empenho em colunas numéricas, como fizemos no treinamento.
    entrada = pd.get_dummies(entrada, columns=["tipo_empenho"], dtype=int)

    # Garante que a consulta tenha exatamente as mesmas colunas usadas no treinamento.
    entrada = entrada.reindex(columns=X.columns, fill_value=0)

    # Pede ao modelo uma previsão para a nota fiscal informada.
    previsao_modelo = modelo.predict(entrada)[0]

    # Aplica as regras do sistema especialista ao mesmo caso.
    resultado_regras = aplicar_regras(caso)

    # Mostra os resultados dos dois componentes.
    print("PREVISÃO DO MODELO:", previsao_modelo)
    print("REGRA APLICADA:", resultado_regras["regra"])
    print("RECOMENDAÇÃO FINAL:", resultado_regras["recomendacao"])
    print("EXPLICAÇÃO:", resultado_regras["explicacao"])

    # Retorna os resultados para que também possam ser utilizados em outras células.
    return {
        "previsao_modelo": previsao_modelo,
        "regra": resultado_regras["regra"],
        "recomendacao": resultado_regras["recomendacao"],
        "explicacao": resultado_regras["explicacao"]
    }

## Célula 14 — Teste de uma consulta

Vamos simular uma nota fiscal em que todas as conferências estão corretas, mas o OD ainda não assinou.

Pelo fluxo definido para o projeto, a recomendação deve ser encaminhar a nota ao ordenador para assinatura — e não devolvê-la ao gestor.

In [ ]:
# Célula 14: dados de uma nota fiscal simulada para consulta

caso_exemplo = {
    "tipo_empenho": "Ordinário",  # Tipo do empenho.
    "valor_confere": True,  # O valor da nota está correto.
    "empenho_confere": True,  # A nota corresponde ao empenho.
    "saldo_suficiente": True,  # Existe saldo suficiente.
    "certidoes_validas": True,  # As certidões estão válidas.
    "ateste_recebimento": True,  # Existe ateste do recebimento.
    "ordenador_assinou": False  # O OD ainda não assinou.
}

# Executa a consulta e guarda o resultado.
resultado_consulta = consultar_sistema(caso_exemplo)

## Célula 15 — Experimente outros cenários

Altere os valores abaixo para simular outras situações.

Sugestões:
- Mude `valor_confere` para `False` e veja a devolução ao gestor.
- Mantenha as conferências corretas e mude `ordenador_assinou` para `True` para encaminhar à liquidação.
- Mude `certidoes_validas` para `False` para simular uma pendência documental.

Use `True` para indicar que a condição foi atendida e `False` para indicar que não foi atendida.

In [ ]:
# Célula 15: altere os valores para testar outras situações

caso_teste = {
    "tipo_empenho": "Global",  # Pode ser Ordinário, Global ou Estimativo.
    "valor_confere": True,  # Altere para False para simular valor incorreto.
    "empenho_confere": True,  # Altere para False para simular empenho incompatível.
    "saldo_suficiente": True,  # Altere para False para simular saldo insuficiente.
    "certidoes_validas": True,  # Altere para False para simular certidões inválidas.
    "ateste_recebimento": True,  # Altere para False para simular falta de ateste.
    "ordenador_assinou": True  # Altere para False para aguardar assinatura do OD.
}

# Executa o sistema com os valores escolhidos.
resultado_teste = consultar_sistema(caso_teste)

## Célula 16 — Conclusão

O projeto apresentou um protótipo de sistema híbrido para apoiar a conferência de notas fiscais antes da liquidação.

Foi criada uma base de dados fictícia com 36 registros e três classificações. Os dados foram separados em treinamento e teste, e uma árvore de decisão foi treinada e avaliada por meio da acurácia, da matriz de confusão e do relatório de classificação.

Também foi desenvolvido um motor de inferência com quatro regras, responsável por verificar as condições da nota fiscal e apresentar uma recomendação acompanhada de explicação.

**Limitações:** a base é pequena, simulada e construída a partir de critérios definidos para fins didáticos. Portanto, os resultados não comprovam o desempenho do sistema em processos reais. Para uma aplicação prática, seria necessário utilizar dados reais autorizados, revisar os critérios com o setor responsável e validar as regras com as normas e procedimentos vigentes.

**Resultado esperado do trabalho:** demonstrar como o aprendizado de máquina pode ser combinado com regras explícitas para apoiar uma decisão administrativa, mantendo a necessidade de revisão humana.